In [5]:
import pandas as pd
import numpy as np
import joblib
import subprocess

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Pandas: 2.3.3
NumPy: 2.5.1
Matplotlib: 3.11.1
Scikit-learn: 1.9.0
Joblib: 1.5.3


In [6]:
# Load the model and preprocessors
model   = joblib.load('freight_rate_model.pkl')
imputer = joblib.load('imputer.pkl')
encoder = joblib.load('equipment_encoder.pkl')

In [7]:
# Load the feature list
with open('feature_columns.txt', 'r') as f:
    FEATURES = f.read().splitlines()

In [8]:
print(f" Model loaded:    {type(model).__name__}")
print(f" Imputer loaded:  {type(imputer).__name__}")
print(f" Encoder loaded:  {type(encoder).__name__}")
print(f" Features loaded: {FEATURES}")

 Model loaded:    RandomForestRegressor
 Imputer loaded:  SimpleImputer
 Encoder loaded:  LabelEncoder
 Features loaded: ['distance', 'weight', 'market_index', 'quote_signal', 'equipment', 'month', 'week', 'day_of_week', 'quarter']


In [9]:
# Load data
df_dec = pd.read_csv('december-chart-inputs.csv')
 
print(f"\nDecember file loaded: {df_dec.shape}")
print(f"Columns: {df_dec.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df_dec.head())
print(f"\nMissing predicted_rate: {df_dec['predicted_rate'].isnull().sum()} rows need to be filled")


December file loaded: (31, 7)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

First 5 rows:
      pickup    delivery  distance equipment  weight        date  \
0  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-05   

   predicted_rate  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN  

Missing predicted_rate: 31 rows need to be filled


In [10]:
# Feature Engineering

df_dec['weight']      = df_dec['weight'].abs()          # fix sign-flips
df_dec['date']        = pd.to_datetime(df_dec['date'])
df_dec['month']       = df_dec['date'].dt.month          # 12 (December)
df_dec['week']        = df_dec['date'].dt.isocalendar().week.astype(int)
df_dec['day_of_week'] = df_dec['date'].dt.dayofweek      # 0=Mon, 6=Sun
df_dec['quarter']     = df_dec['date'].dt.quarter        # 4 (Q4)
 
print(f" Engineered: month, week, day_of_week, quarter")
print(f"\nFeature values for December:")
print(df_dec[['date', 'month', 'week', 'day_of_week', 'quarter']].to_string(index=False))

 Engineered: month, week, day_of_week, quarter

Feature values for December:
      date  month  week  day_of_week  quarter
2025-12-01     12    49            0        4
2025-12-02     12    49            1        4
2025-12-03     12    49            2        4
2025-12-04     12    49            3        4
2025-12-05     12    49            4        4
2025-12-06     12    49            5        4
2025-12-07     12    49            6        4
2025-12-08     12    50            0        4
2025-12-09     12    50            1        4
2025-12-10     12    50            2        4
2025-12-11     12    50            3        4
2025-12-12     12    50            4        4
2025-12-13     12    50            5        4
2025-12-14     12    50            6        4
2025-12-15     12    51            0        4
2025-12-16     12    51            1        4
2025-12-17     12    51            2        4
2025-12-18     12    51            3        4
2025-12-19     12    51            4        4
202

In [11]:
# median imputaion for market index and signal 

print(f"\nFeatures needed:      {FEATURES}")
print(f"Features available:   {df_dec.columns.tolist()}")


Features needed:      ['distance', 'weight', 'market_index', 'quote_signal', 'equipment', 'month', 'week', 'day_of_week', 'quarter']
Features available:   ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate', 'month', 'week', 'day_of_week', 'quarter']


In [12]:
# Use training medians for the two missing features
df_dec['market_index'] = imputer.statistics_[1]   # median learned during training
df_dec['quote_signal'] = 2.05                      # median from training data
 
print(f"\n market_index filled with training median: {imputer.statistics_[1]:.4f}")
print(f" quote_signal filled with training median: 2.05")


 market_index filled with training median: 1.0559
 quote_signal filled with training median: 2.05


In [15]:
# Selecting Features and Preprocess

X_dec = df_dec[FEATURES].copy()
 
print(f"\nX_dec shape: {X_dec.shape}")
print(f"X_dec columns: {X_dec.columns.tolist()}")
print(f"Missing values before preprocessing: {X_dec.isnull().sum().sum()}")
 
# Apply SAME preprocessors — transform() only, NEVER fit_transform()!
IMPUTE_COLS = ['weight', 'market_index']
X_dec[IMPUTE_COLS]  = imputer.transform(X_dec[IMPUTE_COLS])  # transform only!
X_dec['equipment']  = encoder.transform(X_dec['equipment'])   # transform only!
 
print(f"Equipment encoded: Dry Van: {encoder.transform(['Dry Van'])[0]}")
print(f"Missing values after preprocessing: {X_dec.isnull().sum().sum()}")


X_dec shape: (31, 9)
X_dec columns: ['distance', 'weight', 'market_index', 'quote_signal', 'equipment', 'month', 'week', 'day_of_week', 'quarter']
Missing values before preprocessing: 0
Equipment encoded: Dry Van: 0
Missing values after preprocessing: 0


In [16]:
# December Predictions

predictions = model.predict(X_dec)
 
print(f" Generated {len(predictions)} December predictions")
print(f"\n  Min rate:    ${predictions.min():.2f}")
print(f"  Max rate:    ${predictions.max():.2f}")
print(f"  Mean rate:   ${predictions.mean():.2f}")
print(f"  Median rate: ${np.median(predictions):.2f}")

 Generated 31 December predictions

  Min rate:    $746.20
  Max rate:    $818.60
  Mean rate:   $811.05
  Median rate: $818.18


In [17]:
# Fill in the December file and save

df_output = pd.read_csv('december-chart-inputs.csv')
df_output['predicted_rate'] = predictions.round(2)
 
# Keep only the 7 required columns in the required order
output_cols = ['pickup', 'delivery', 'distance', 'equipment',
               'weight', 'date', 'predicted_rate']
df_output = df_output[output_cols]
 
print(f"\nFilled december file:")
print(df_output.to_string(index=False))
 
# Save the completed file
df_output.to_csv('december_chart_inputs_filled.csv', index=False)
print(f"\n✓ Saved: december_chart_inputs_filled.csv")


Filled december file:
   pickup   delivery  distance equipment  weight       date  predicted_rate
Lexington Fort Wayne       360   Dry Van   32000 2025-12-01          818.18
Lexington Fort Wayne       360   Dry Van   32000 2025-12-02          818.52
Lexington Fort Wayne       360   Dry Van   32000 2025-12-03          818.54
Lexington Fort Wayne       360   Dry Van   32000 2025-12-04          818.60
Lexington Fort Wayne       360   Dry Van   32000 2025-12-05          818.10
Lexington Fort Wayne       360   Dry Van   32000 2025-12-06          817.87
Lexington Fort Wayne       360   Dry Van   32000 2025-12-07          816.13
Lexington Fort Wayne       360   Dry Van   32000 2025-12-08          818.18
Lexington Fort Wayne       360   Dry Van   32000 2025-12-09          818.52
Lexington Fort Wayne       360   Dry Van   32000 2025-12-10          818.54
Lexington Fort Wayne       360   Dry Van   32000 2025-12-11          818.60
Lexington Fort Wayne       360   Dry Van   32000 2025-12-12      